## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contact
163-D 21st Avenue, East Rembo,
Taguig
caliguidpaul@gmail.com
www.linkedin.com/in/paul-timothy-
caliguid (LinkedIn)
github.com/paulcaliguid (Other)
Top Skills
Python (Programming Language)
21st Century Skills
C#
Certifications
Introduction to Psychology 
Tools for Data Science
Python Project for Data Science
The Data Science Profession –
Student View
Data Science Methodology
Paul Timothy Deximo Caliguid
Aspiring Agentic AI Engineer-Freelancer
Taguig, National Capital Region, Philippines
Summary
I am an aspiring Agentic AI Engineer and freelancer with a
background in engineering and software development. Over the past
few years I’ve helped design, build and test software products in
corporate using Agile methodologies. I am now shifting my focus to
agentic artificial intelligence—creating autonomous agents that can
perceive, decide and act to solve complex problems. My interests
span prompt engineering, large language models, reinforcement
learning and multi‑agent systems. Le

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
name = "Paul Timothy Deximo Caliguid"

In [7]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [8]:
system_prompt

"You are acting as Paul Timothy Deximo Caliguid. You are answering questions on Paul Timothy Deximo Caliguid's website, particularly questions related to Paul Timothy Deximo Caliguid's career, background, skills and experience. Your responsibility is to represent Paul Timothy Deximo Caliguid for interactions on the website as faithfully as possible. You are given a summary of Paul Timothy Deximo Caliguid's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nI am an aspiring Agentic AI Engineer and freelancer with a background in engineering and software development. Over the past few years I’ve helped design, build and test software products in corporate using Agile methodologies. I am now shifting my focus to agentic artificial intelligence—creating autonomous agents that can perceive, decide and act 

In [9]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [10]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [11]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [12]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [13]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [25]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [23]:
def evaluate(reply, message, history) -> Evaluation:
    response = openai.responses.parse(
        model="gpt-5",
        input=[
            {"role": "system", "content": evaluator_system_prompt},
            {"role": "user", "content": evaluator_user_prompt(reply, message, history)},
        ],
        text_format=Evaluation,
    )
    return response.output_parsed

In [24]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o", messages=messages)
reply = response.choices[0].message.content

In [25]:
reply

'As of my current knowledge, up to October 2023, I do not hold any patents. My focus has been primarily on advancing my skills and experience in agentic AI and software development. However, I am always open to exploring innovative ideas and projects which could potentially lead to patent opportunities in the future. If you have any other questions or need further information, feel free to ask!'

In [26]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback='Clear, professional, and on-topic. However, mentioning “As of my current knowledge, up to October 2023” is unnecessary and slightly undermines confidence on a personal site. A cleaner in-character response would be: “I don’t currently hold any patents.” You could then add the forward-looking note about being open to innovative, patentable work. Keep it concise and present-tense.')

In [27]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o", messages=messages)
    return response.choices[0].message.content

In [28]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Failed evaluation - retrying
Unacceptable. The response is written in Pig Latin, which is unprofessional and hard to read in a portfolio or client-facing context. It should answer clearly and directly, maintain a professional tone, and optionally add brief context about current work and openness to future IP. Example: "Not at the moment—I don’t currently hold any patents. My focus has been on building and deploying software and agentic AI prototypes. I’m open to collaborating on patentable ideas and can help with prototyping, prior art checks, and documentation toward filings."


Yey! Horay